<a href="https://colab.research.google.com/github/5ahar-K/CodeGraph-agent/blob/main/Return_value_Tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install networkx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.3 MB/s eta 0:00:00


In [27]:
!git clone https://github.com/pallets/click.git target_repo

fatal: destination path 'target_repo' already exists and is not an empty directory.


In [12]:
#Messy
!git clone https://github.com/emilybache/GildedRose-Refactoring-Kata messy_repo2
!git clone https://github.com/christianhujer/expensereport messy_repo
!git clone https://github.com/mailpile/Mailpile messy_repo1

Cloning into 'messy_repo2'...
remote: Enumerating objects: 5187, done.
remote: Total 5187 (delta 0), reused 0 (delta 0), pack-reused 5187 (from 1)
Receiving objects: 100% (5187/5187), 3.61 MiB | 13.68 MiB/s, done.
Resolving deltas: 100% (1926/1926), done.
Cloning into 'messy_repo'...
remote: Enumerating objects: 1108, done.
remote: Counting objects: 100% (373/373), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 1108 (delta 318), reused 290 (delta 282), pack-reused 735 (from 1)
Receiving objects: 100% (1108/1108), 488.86 KiB | 9.22 MiB/s, done.
Resolving deltas: 100% (423/423), done.
Cloning into 'messy_repo1'...
remote: Enumerating objects: 46438, done.
remote: Total 46438 (delta 0), reused 0 (delta 0), pack-reused 46438 (from 1)
Receiving objects: 100% (46438/46438), 25.25 MiB | 20.39 MiB/s, done.
Resolving deltas: 100% (32007/32007), done.


In [3]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
client = genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [17]:
'Return-value Tracking'
def track_return_flow(func_node):
    assigned_from_call = {}

    for node in ast.walk(func_node):
        if isinstance(node, ast.Assign):
            if isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Name):
                if isinstance(node.targets[0], ast.Name):
                    var_name = node.targets[0].id
                    called_func = node.value.func.id
                    assigned_from_call[var_name] = called_func

    flows = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            target_func = node.func.id
            for arg in node.args:
                if isinstance(arg, ast.Name) and arg.id in assigned_from_call:   #checks if the name of the arg is in the list of function node names
                    source_func = assigned_from_call[arg.id]
                    if target_func != source_func:
                        flows.append((source_func, target_func))
    return flows
# To be used as the third pass in the build_graph function
def add_data_flow_edges(graph, function_bodies, name_to_keys, file_imports):
    for key, func_node in function_bodies.items():
        source_file = key[0]
        imports_in_this_file = file_imports.get(source_file, {})
        flows = track_return_flow(func_node)

        for source_name, target_name in flows:
            source_candidates = name_to_keys.get(source_name)
            target_candidates = name_to_keys.get(target_name)
            if not source_candidates or not target_candidates:
                continue  # one side is a built-in or unresolvable, skip

            # Reuse the exact same same-file-first heuristic
            source_key = next((c for c in source_candidates if c[0] == source_file), source_candidates[0])
            target_key = next((c for c in target_candidates if c[0] == source_file), target_candidates[0])

            graph.add_edge(source_key, target_key, type="data_flow")

    return graph

In [48]:
ambiguous_calls = 0
resolved_to_one = 0
still_ambiguous = 0

for key, func_node in function_bodies.items():
    for called_name in find_function_calls(func_node):
        candidates = name_to_keys.get(called_name)
        if not candidates or len(candidates) <= 1:
            continue  # not ambiguous to begin with, skip entirely

        ambiguous_calls += 1

        # How many of those candidates actually got an edge in the real graph?
        actual_targets = [c for c in candidates if gra.has_edge(key, c)]

        if len(actual_targets) == 1:
            resolved_to_one += 1
        elif len(actual_targets) > 1:
            still_ambiguous += 1

print(f"Calls where multiple same-named functions existed: {ambiguous_calls}")
print(f"  Narrowed down to exactly one target: {resolved_to_one}")
print(f"  Still connected to multiple targets: {still_ambiguous}")

Calls where multiple same-named functions existed: 4163
  Narrowed down to exactly one target: 544
  Still connected to multiple targets: 3619


In [82]:
import ast
import os
import networkx as nx
from collections import defaultdict

def find_function_calls(func_node):
    calls = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                calls.append(node.func.id)
            elif isinstance(node.func, ast.Attribute):
                calls.append(node.func.attr)
    return calls

def get_imports(tree):
    imports = {}
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            module = node.module or ""     #from . import echo    (importing from the current package)
            for alias in node.names:     #e.g. from click.utils import echo as e
                local_name = alias.asname or alias.name
                imports[local_name] = module
        elif isinstance(node, ast.Import):
            for alias in node.names:
                local_name = alias.asname or alias.name    #e.g import click (no from package)
                imports[local_name] = alias.name
    return imports

def module_matches_file(module, filepath):
    if not module:
        return False
    module_as_path = module.replace(".", os.sep)
    normalized_filepath = filepath.replace("\\", os.sep)
    return module_as_path in normalized_filepath

def add_parents(node):
    for child in ast.iter_child_nodes(node):
        child.parent = node
        add_parents(child)

def build_graph(repo_path):
    graph = nx.DiGraph()
    name_to_keys = defaultdict(list)
    function_bodies = {}       # (file, name) -> AST node
    file_imports = {}          # file -> {imported_name: module_string}

    # Pass 1: Find all the functions
    for root, _, files in os.walk(repo_path):
        for filename in files:
            if not filename.endswith(".py"):
                continue
            filepath = os.path.join(root, filename)
            try:
                with open(filepath, encoding="utf-8") as f:
                    tree = ast.parse(f.read(), filename=filepath)
                add_parents(tree) # Add parent pointers to the AST nodes
            except SyntaxError:
                continue

            file_imports[filepath] = get_imports(tree)

            for item in tree.body:
                if isinstance(item, ast.ClassDef):
                    class_name = item.name
                    for sub_item in item.body:
                        if isinstance(sub_item, ast.FunctionDef):
                            key = (filepath, class_name, sub_item.name)
                            graph.add_node(key, name=sub_item.name, file=filepath,
                                           line=sub_item.lineno, class_name=class_name)
                            name_to_keys[sub_item.name].append(key)
                            function_bodies[key] = sub_item

                elif isinstance(item, ast.FunctionDef):
                    key = (filepath, None, item.name)
                    graph.add_node(key, name=item.name, file=filepath,
                                   line=item.lineno, class_name=None)
                    name_to_keys[item.name].append(key)
                    function_bodies[key] = item

    # Pass 2: map each call to the most likely specific function
    for key, func_node in function_bodies.items():
        source_file, _, _ = key
        imports_in_this_file = file_imports.get(source_file, {})

        for called_name in find_function_calls(func_node):
            candidates = name_to_keys.get(called_name)
            if not candidates:
                continue

            resolved = None

            # 1. Same-file candidates win first (any class, or top-level)
            same_file_candidates = [c for c in candidates if c[0] == source_file]
            if same_file_candidates:
                resolved = same_file_candidates

            # 2. Explicitly imported
            elif called_name in imports_in_this_file:
                module = imports_in_this_file[called_name]
                matched = [c for c in candidates if module_matches_file(module, c[0])]
                if matched:
                    resolved = matched

            # 3. Only one candidate anywhere
            if resolved is None and len(candidates) == 1:
                resolved = candidates

            # 4. Still ambiguous
            if resolved is None:
                resolved = candidates

            for target_key in resolved:
                graph.add_edge(key, target_key)

    # Pass 3: Data flow edges
    graph = add_data_flow_edges(graph, function_bodies, name_to_keys, file_imports)
    return graph, function_bodies, name_to_keys
graph, function_bodies, name_to_keys = build_graph("messy_repo1")
#graph, function_bodies, name_to_keys = build_graph("target_repo")
print(f"Found {graph.number_of_nodes()} functions, {graph.number_of_edges()} call relationships")
ambiguous_count = 0
for key, func_node in function_bodies.items():
    calls = find_function_calls(func_node)
    for name in calls:
        if len(name_to_keys.get(name, [])) > 1:
           ambiguous_count += 1

print(f"Calls to ambiguously-named functions: {ambiguous_count}")

messy_repo1/mailpile/app.py:96: SyntaxWarning: invalid escape sequence '\S'
  elif re.search(' >\S+$', opt):
messy_repo1/mailpile/app.py:101: SyntaxWarning: invalid escape sequence '\S'
  if re.search(' :(json|j?html|text|\S+.html(?:!\S+))$', opt):
messy_repo1/mailpile/search.py:1505: SyntaxWarning: invalid escape sequence '\s'
  '(\n[^\s]+ [^\n]+(@[^\n]+|(wrote|crit|schreib)):\s+>[^\n]+'
messy_repo1/mailpile/search.py:1508: SyntaxWarning: invalid escape sequence '\s'
  '|\n--[^\n]+BEGIN PGP[^\n]+--\s+(\S+:[^\n]+\n)*'  # PGP header
messy_repo1/mailpile/search.py:1510: SyntaxWarning: invalid escape sequence '\s'
  SNIPPET_SPACE_RE = re.compile('\s+')
messy_repo1/mailpile/util.py:61: SyntaxWarning: invalid escape sequence '\s'
  WORD_REGEXP = re.compile('[^\s!@#$%^&*\(\)_+=\{\}\[\]'
messy_repo1/mailpile/util.py:62: SyntaxWarning: invalid escape sequence '\<'
  ':\"|;`\'\\\<\>\?,\.\/\-]{2,}')
messy_repo1/mailpile/util.py:116: SyntaxWarning: invalid escape sequence '\A'
  ALPHA_RE  = re.co

Found 2538 functions, 15118 call relationships
Calls to ambiguously-named functions: 4163


In [83]:
def what_calls(graph, filepath, class_name, function_name):
    key = (filepath, class_name, function_name)
    return list(graph.predecessors(key))

def what_does_it_call(graph, filepath, class_name, function_name):
    key = (filepath, class_name, function_name)
    return list(graph.successors(key))

In [84]:
def find_matches(graph, function_name):
    #Return all (file, name) keys in the graph matching this function name.
    return [key for key in graph.nodes if key[2] == function_name]

In [85]:
graph, _, _ = build_graph("target_repo")
matches = find_matches(graph, "command")
for m in matches:
    print(m)

('target_repo/src/click/core.py', 'Group', 'command')
('target_repo/src/click/decorators.py', None, 'command')


In [86]:
def add_parent_links(tree):
    """Give every node in the tree a .parent attribute pointing to its parent node."""
    for node in ast.walk(tree):
        for child in ast.iter_child_nodes(node):
            child.parent = node
    return tree

In [87]:
def get_function_source(graph, key):
    filepath, class_name, function_name = key
    with open(filepath, encoding="utf-8") as f:
        source_text = f.read()
    tree = ast.parse(source_text)
    add_parent_links(tree)
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            # If it's a method, ensure we get the right one by checking class_name
            if class_name and isinstance(node.parent, ast.ClassDef) and node.parent.name == class_name:
                return ast.get_source_segment(source_text, node)
            # If it's a top-level function, class_name will be None
            elif not class_name and not isinstance(node.parent, ast.ClassDef):
                return ast.get_source_segment(source_text, node)
    return None

def ask_about_function(graph, key, question):
    source = get_function_source(graph, key)
    filepath, _, function_name = key
    callers = what_calls(graph, filepath, function_name)
    callees = what_does_it_call(graph, filepath, function_name)

    prompt = f"""Here is a function called `{function_name}` from file `{filepath}`:

{source}

It is called by: {callers}
It calls: {callees}

Question: {question}

Answer using only the information given above. If you can't determine the answer from this, say so."""

    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

def generate_test(graph, key):
    source = get_function_source(graph, key)
    prompt = f"""Write a pytest unit test for this function:

{source}

Only output the test code, no explanation."""
    model = client.GenerativeModel(model_name="gemini-flash-latest")
    response = model.generate_content(contents=prompt)
    return response.text

In [93]:
g2, function_bodies2, name_to_keys2 = build_graph("messy_repo1")
print(f"Found {g2.number_of_nodes()} functions, {g2.number_of_edges()} call relationships")
for node in g2.nodes(data=True):
     print(node)

messy_repo1/mailpile/app.py:96: SyntaxWarning: invalid escape sequence '\S'
  elif re.search(' >\S+$', opt):
messy_repo1/mailpile/app.py:101: SyntaxWarning: invalid escape sequence '\S'
  if re.search(' :(json|j?html|text|\S+.html(?:!\S+))$', opt):
messy_repo1/mailpile/search.py:1505: SyntaxWarning: invalid escape sequence '\s'
  '(\n[^\s]+ [^\n]+(@[^\n]+|(wrote|crit|schreib)):\s+>[^\n]+'
messy_repo1/mailpile/search.py:1508: SyntaxWarning: invalid escape sequence '\s'
  '|\n--[^\n]+BEGIN PGP[^\n]+--\s+(\S+:[^\n]+\n)*'  # PGP header
messy_repo1/mailpile/search.py:1510: SyntaxWarning: invalid escape sequence '\s'
  SNIPPET_SPACE_RE = re.compile('\s+')
messy_repo1/mailpile/util.py:61: SyntaxWarning: invalid escape sequence '\s'
  WORD_REGEXP = re.compile('[^\s!@#$%^&*\(\)_+=\{\}\[\]'
messy_repo1/mailpile/util.py:62: SyntaxWarning: invalid escape sequence '\<'
  ':\"|;`\'\\\<\>\?,\.\/\-]{2,}')
messy_repo1/mailpile/util.py:116: SyntaxWarning: invalid escape sequence '\A'
  ALPHA_RE  = re.co

Found 2538 functions, 15118 call relationships
(('messy_repo1/install_hooks.py', None, 'symlink_develop'), {'name': 'symlink_develop', 'file': 'messy_repo1/install_hooks.py', 'line': 4, 'class_name': None})
(('messy_repo1/setup.py', None, '_get_submodules'), {'name': '_get_submodules', 'file': 'messy_repo1/setup.py', 'line': 24, 'class_name': None})
(('messy_repo1/setup.py', None, '_find_git_files'), {'name': '_find_git_files', 'file': 'messy_repo1/setup.py', 'line': 31, 'class_name': None})
(('messy_repo1/setup.py', 'Builder', 'run'), {'name': 'run', 'file': 'messy_repo1/setup.py', 'line': 68, 'class_name': 'Builder'})
(('messy_repo1/mailpile/eventlog.py', None, 'NewEventId'), {'name': 'NewEventId', 'file': 'messy_repo1/mailpile/eventlog.py', 'line': 22, 'class_name': None})
(('messy_repo1/mailpile/eventlog.py', None, '_ClassName'), {'name': '_ClassName', 'file': 'messy_repo1/mailpile/eventlog.py', 'line': 34, 'class_name': None})
(('messy_repo1/mailpile/eventlog.py', 'Event', 'Parse'

In [94]:
while True:
    name = input("\nEnter a function name (or 'quit'): ")
    if name == "quit":
        break
    matches = find_matches(g2, name)
    if not matches:
        print("Function not found in graph.")
        continue
    if len(matches) > 1:
        print(f"Multiple functions named '{name}' found:")
        for i, m in enumerate(matches):
            print(f"  [{i}] {m[0]}")
        idx = int(input("Which one? Enter the number: "))
        key = matches[idx]
    else:
        key = matches[0]

    q = input("Your question (or type 'test' to generate a unit test): ")
    if q == "test":
        print(generate_test(g2, key))
    else:
        print(ask_about_function(g2, key, q))


Enter a function name (or 'quit'): command#
Function not found in graph.

Enter a function name (or 'quit'): command
Multiple functions named 'command' found:
  [0] messy_repo1/mailpile/auth.py
  [1] messy_repo1/mailpile/auth.py
  [2] messy_repo1/mailpile/auth.py
  [3] messy_repo1/mailpile/app.py
  [4] messy_repo1/mailpile/app.py
  [5] messy_repo1/mailpile/plugins/setup_magic.py
  [6] messy_repo1/mailpile/plugins/compose.py
  [7] messy_repo1/mailpile/plugins/compose.py
  [8] messy_repo1/mailpile/plugins/compose.py
  [9] messy_repo1/mailpile/plugins/compose.py
  [10] messy_repo1/mailpile/plugins/compose.py
  [11] messy_repo1/mailpile/plugins/compose.py
  [12] messy_repo1/mailpile/plugins/compose.py
  [13] messy_repo1/mailpile/plugins/compose.py
  [14] messy_repo1/mailpile/plugins/compose.py
  [15] messy_repo1/mailpile/plugins/compose.py
  [16] messy_repo1/mailpile/plugins/motd.py
  [17] messy_repo1/mailpile/plugins/vcard_gnupg.py
  [18] messy_repo1/mailpile/plugins/vcard_gnupg.py
  [19

In [64]:
# 1. How many files failed to parse entirely?
parse_failures = []
for root, _, files in os.walk("messy_repo1"):
    for filename in files:
        if not filename.endswith(".py"):
            continue
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, encoding="utf-8") as f:
                ast.parse(f.read(), filename=filepath)
        except SyntaxError as e:
            parse_failures.append((filepath, str(e)))

print(f"Files that failed to parse: {len(parse_failures)}")
for fp, err in parse_failures[:10]:
    print(" -", fp, "|", err)

# 2. Which names are ambiguous, and how ambiguous?
ambiguous_names = {name: keys for name, keys in name_to_keys2.items() if len(keys) > 1}
print(f"\nAmbiguous function names: {len(ambiguous_names)}")
# Sort by how many duplicates exist, worst first
worst = sorted(ambiguous_names.items(), key=lambda kv: -len(kv[1]))
for name, keys in worst[:10]:
    print(f" - '{name}' defined in {len(keys)} places:")
    for k in keys:
        print("     ", k)

# 3. Calls that still fell back to connect to everything (unresolved by the heuristic)
still_ambiguous_calls = []
for key, func_node in function_bodies2.items():
    for called_name in find_function_calls(func_node):
        candidates = name_to_keys2.get(called_name)
        if candidates and len(candidates) > 1:
            actual_targets = [c for c in candidates if g2.has_edge(key, c)]
            if len(actual_targets) > 1:
                still_ambiguous_calls.append((key, called_name, actual_targets))

print(f"\nCalls the heuristic couldn't fully resolve: {len(still_ambiguous_calls)}")
for caller, name, targets in still_ambiguous_calls[:10]:
    print(f" - {caller} calls '{name}', still ambiguous between {len(targets)} candidates")

messy_repo1/mailpile/app.py:96: SyntaxWarning: invalid escape sequence '\S'
  elif re.search(' >\S+$', opt):
messy_repo1/mailpile/app.py:101: SyntaxWarning: invalid escape sequence '\S'
  if re.search(' :(json|j?html|text|\S+.html(?:!\S+))$', opt):
messy_repo1/mailpile/search.py:1505: SyntaxWarning: invalid escape sequence '\s'
  '(\n[^\s]+ [^\n]+(@[^\n]+|(wrote|crit|schreib)):\s+>[^\n]+'
messy_repo1/mailpile/search.py:1508: SyntaxWarning: invalid escape sequence '\s'
  '|\n--[^\n]+BEGIN PGP[^\n]+--\s+(\S+:[^\n]+\n)*'  # PGP header
messy_repo1/mailpile/search.py:1510: SyntaxWarning: invalid escape sequence '\s'
  SNIPPET_SPACE_RE = re.compile('\s+')
messy_repo1/mailpile/util.py:61: SyntaxWarning: invalid escape sequence '\s'
  WORD_REGEXP = re.compile('[^\s!@#$%^&*\(\)_+=\{\}\[\]'
messy_repo1/mailpile/util.py:62: SyntaxWarning: invalid escape sequence '\<'
  ':\"|;`\'\\\<\>\?,\.\/\-]{2,}')
messy_repo1/mailpile/util.py:116: SyntaxWarning: invalid escape sequence '\A'
  ALPHA_RE  = re.co

Files that failed to parse: 8
 - messy_repo1/mailpile/urlmap.py | invalid syntax (urlmap.py, line 79)
 - messy_repo1/mailpile/conn_brokers.py | invalid syntax (conn_brokers.py, line 705)
 - messy_repo1/mailpile/util.py | invalid syntax (util.py, line 1153)
 - messy_repo1/mailpile/commands.py | invalid syntax (commands.py, line 217)
 - messy_repo1/mailpile/postinglist.py | inconsistent use of tabs and spaces in indentation (postinglist.py, line 131)
 - messy_repo1/mailpile/tests/data/pgp-data/buildexamples.py | inconsistent use of tabs and spaces in indentation (buildexamples.py, line 30)
 - messy_repo1/mailpile/plugins/crypto_autocrypt.py | Missing parentheses in call to 'print'. Did you mean print(...)? (crypto_autocrypt.py, line 621)
 - messy_repo1/scripts/minimize-pgp-key.py | Missing parentheses in call to 'print'. Did you mean print(...)? (minimize-pgp-key.py, line 14)

Ambiguous function names: 219
 - '__init__' defined in 137 places:
      ('messy_repo1/mailpile/eventlog.py', 'E

In [65]:
'Dummy Test'
import ast
import networkx as nx
from collections import defaultdict

dummy_code = """
def fetch_data():
    return {"a": 1}

def clean(data):
    return data

def save(data):
    print(data)

def process():
    result = fetch_data()
    cleaned = clean(result)
    save(cleaned)
"""

fake_filepath = "dummy.py"
tree = ast.parse(dummy_code)

graph = nx.DiGraph()
name_to_keys = defaultdict(list)
function_bodies = {}
file_imports = {fake_filepath: {}}  # no imports in this dummy file

for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        key = (fake_filepath, None, node.name)  # None = no class
        graph.add_node(key, name=node.name)
        name_to_keys[node.name].append(key)
        function_bodies[key] = node

g = add_data_flow_edges(graph, function_bodies, name_to_keys, file_imports)


data_flow_edges = [(u, v) for u, v, d in g.edges(data=True) if d.get("type") == "data_flow"]
print(f"Found {len(data_flow_edges)} data-flow edges")
for u, v in data_flow_edges:
    print(u[2], "->", v[2])  # just print function names

Found 2 data-flow edges
fetch_data -> clean
clean -> save


In [66]:
data_flow_edges = [(u, v) for u, v, d in gra.edges(data=True) if d.get("type") == "data_flow"]
print(f"Found {len(data_flow_edges)} data-flow edges out of {gra.number_of_edges()} total edges")

Found 41 data-flow edges out of 15118 total edges


In [67]:
'Blast radius'
import networkx as nx

def blast_radius(graph, key):
    #How many functions depend on this one?
    return len(nx.ancestors(graph, key))

def top_blast_radius(graph, n=10):
    #Rank every function in the graph by blast radius, highest first.
    scores = [(key, len(nx.ancestors(graph, key))) for key in graph.nodes]
    scores.sort(key=lambda x: -x[1])
    return scores[:n]

for key, score in top_blast_radius(gra, n=10):
    print(score, key)

1681 ('messy_repo1/mailpile/vfs.py', 'FilePath', 'join')
1680 ('messy_repo1/mailpile/crypto/streamer.py', 'IOFilter', 'join')
1655 ('messy_repo1/mailpile/packing.py', None, 'PackIntSet')
1655 ('messy_repo1/mailpile/packing.py', None, 'PackLongList')
1654 ('messy_repo1/mailpile/packing.py', 'StorageBackedData', '_pack')
1654 ('messy_repo1/mailpile/packing.py', 'StorageBackedSet', '_pack')
1654 ('messy_repo1/mailpile/packing.py', 'StorageBackedLongs', '_pack')
1653 ('messy_repo1/mailpile/packing.py', 'StorageBackedData', 'save')
1652 ('messy_repo1/mailpile/packing.py', 'StorageBackedData', '_dirty_maybe_save')
1650 ('messy_repo1/mailpile/vfs.py', 'FilePath', 'startswith')
